# Experiment: FEniCSx Five-Hole Navier-Stokes Dataset Visualization

目标：
- 读取 `ns_2d_fenicsx.py` 生成的 `.mat` 数据集。
- 快速检查五孔几何、初始涡量、轨迹快照和数据统计是否合理。
- 生成可复用的图片输出，便于后续训练前做 sanity check。


In [ ]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint

import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import scipy.io

SEED = 7
rng = np.random.default_rng(SEED)

REPO_ROOT = Path("/Users/zhougc/Desktop/IID/LRTOR_project/Bias_Aware_FNO")
DATA_DIR = REPO_ROOT / "data"
NOTEBOOK_DIR = REPO_ROOT / "output" / "jupyter-notebook"
ARTIFACT_DIR = NOTEBOOK_DIR / "ns-5holes-fenicsx-dataset-visualization"
FIG_DIR = ARTIFACT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

GEN_CMD = (
    "/Users/zhougc/miniconda3/envs/fenicsx-env/bin/python "
    "data_generation/navier_stokes_5holes/ns_2d_fenicsx.py "
    "--grid-resolution 64 --n-train 2 --n-test 1 --record-steps 10 "
    "--final-time 1.0 --dt 0.1 --mesh-size-min 0.10 --mesh-size-max 0.14"
)


def latest_dataset(split: str) -> Path | None:
    files = sorted(DATA_DIR.glob(f"ns_5holes_*_{split}.mat"))
    return files[-1] if files else None


def load_dataset(path: Path) -> dict[str, np.ndarray]:
    try:
        raw = scipy.io.loadmat(path)
        payload = {key: value for key, value in raw.items() if not key.startswith("__")}
    except Exception:
        payload = {}
        with h5py.File(path, "r") as handle:
            layout = handle.attrs.get("bias_aware_layout", "")
            if isinstance(layout, bytes):
                layout = layout.decode("utf-8")
            for key in handle.keys():
                value = handle[key][()]
                if layout != "numpy":
                    value = np.transpose(value, axes=range(len(value.shape) - 1, -1, -1))
                payload[key] = value
    payload["path"] = Path(path)
    return payload


def draw_field(ax, field: np.ndarray, title: str, theta: np.ndarray, cmap: str = "coolwarm", vmin=None, vmax=None):
    image = ax.imshow(field.T, origin="lower", extent=(0.0, 1.0, 0.0, 1.0), cmap=cmap, vmin=vmin, vmax=vmax)
    for cx, cy, radius in theta:
        ax.add_patch(patches.Circle((float(cx), float(cy)), float(radius), fill=False, ec="white", lw=1.2))
    ax.set_title(title)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")
    return image


def describe_payload(payload: dict[str, np.ndarray]) -> dict[str, object]:
    summary = {
        "path": str(payload["path"]),
        "a_shape": tuple(np.asarray(payload["a"]).shape),
        "u_shape": tuple(np.asarray(payload["u"]).shape),
        "mask_shape": tuple(np.asarray(payload["mask"]).shape),
        "theta_shape": tuple(np.asarray(payload["theta"]).shape),
        "t_shape": tuple(np.asarray(payload["t"]).shape),
        "has_aux_fields": all(name in payload for name in ("velocity_u", "velocity_v", "pressure")),
    }
    if "nu" in payload:
        summary["nu"] = float(np.asarray(payload["nu"]).reshape(-1)[0])
    if "eta" in payload:
        summary["eta"] = float(np.asarray(payload["eta"]).reshape(-1)[0])
    if "gamma_drive" in payload:
        summary["gamma_drive"] = float(np.asarray(payload["gamma_drive"]).reshape(-1)[0])
    return summary


print(f"Data directory: {DATA_DIR}")
print(f"Notebook artifacts: {ARTIFACT_DIR}")
print(f"Suggested generation command:\n{GEN_CMD}")


## Data Discovery

这个 notebook 不负责生成数据，只读取 `data/` 下已经存在的 `.mat` 文件。
如果这里报 `FileNotFoundError`，先运行上面的 FEniCSx 生成命令。


In [ ]:
TRAIN_PATH = latest_dataset("train")
TEST_PATH = latest_dataset("test")
OOD_PATH = latest_dataset("ood")

if TRAIN_PATH is None:
    raise FileNotFoundError(
        "No train dataset was found under data/. Run the generator first, for example:\n"
        f"{GEN_CMD}"
    )

datasets = {
    "train": load_dataset(TRAIN_PATH),
    "test": load_dataset(TEST_PATH) if TEST_PATH is not None else None,
    "ood": load_dataset(OOD_PATH) if OOD_PATH is not None else None,
}

summary = {name: describe_payload(payload) for name, payload in datasets.items() if payload is not None}
pprint(summary)


## Dataset Overview

下面先看一个样本：障碍物 mask、初始涡量 `a`，以及若干时刻的涡量快照。


In [ ]:
SPLIT_NAME = "train"
SAMPLE_IDX = 0

payload = datasets[SPLIT_NAME]
a = np.asarray(payload["a"])
u = np.asarray(payload["u"])
mask = np.asarray(payload["mask"]).astype(bool)
theta = np.asarray(payload["theta"])
time_grid = np.asarray(payload["t"]).reshape(-1)

frame_ids = np.unique(np.array([0, len(time_grid) // 2, len(time_grid) - 1], dtype=int))
fields = [a[SAMPLE_IDX], *[u[SAMPLE_IDX, :, :, frame_id] for frame_id in frame_ids]]
limit = max(float(np.max(np.abs(field))) for field in fields)
limit = max(limit, 1e-6)

fig, axes = plt.subplots(1, 2 + len(frame_ids), figsize=(4.2 * (2 + len(frame_ids)), 4.2), constrained_layout=True)
draw_field(axes[0], mask[SAMPLE_IDX].astype(float), f"{SPLIT_NAME} sample {SAMPLE_IDX}: mask", theta[SAMPLE_IDX], cmap="gray_r", vmin=0.0, vmax=1.0)
draw_field(axes[1], a[SAMPLE_IDX], "initial vorticity a", theta[SAMPLE_IDX], vmin=-limit, vmax=limit)
for ax, frame_id in zip(axes[2:], frame_ids):
    draw_field(ax, u[SAMPLE_IDX, :, :, frame_id], f"w(t={time_grid[frame_id]:.2f})", theta[SAMPLE_IDX], vmin=-limit, vmax=limit)

save_path = FIG_DIR / f"{SPLIT_NAME}_sample_{SAMPLE_IDX}_overview.png"
fig.savefig(save_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved figure -> {save_path}")


## Sample Gallery

这一部分适合快速检查多样本几何和轨迹末态是否明显异常。


In [ ]:
payload = datasets["train"]
a = np.asarray(payload["a"])
u = np.asarray(payload["u"])
mask = np.asarray(payload["mask"]).astype(bool)
theta = np.asarray(payload["theta"])
time_grid = np.asarray(payload["t"]).reshape(-1)

gallery_size = min(4, a.shape[0])
sample_ids = np.linspace(0, a.shape[0] - 1, gallery_size, dtype=int)
fig, axes = plt.subplots(gallery_size, 4, figsize=(16, 4 * gallery_size), constrained_layout=True)
if gallery_size == 1:
    axes = np.asarray([axes])

for row, sample_id in enumerate(sample_ids):
    local_fields = [a[sample_id], u[sample_id, :, :, 0], u[sample_id, :, :, -1]]
    local_limit = max(float(np.max(np.abs(field))) for field in local_fields)
    local_limit = max(local_limit, 1e-6)
    draw_field(axes[row, 0], mask[sample_id].astype(float), f"sample {sample_id}: mask", theta[sample_id], cmap="gray_r", vmin=0.0, vmax=1.0)
    draw_field(axes[row, 1], a[sample_id], "initial a", theta[sample_id], vmin=-local_limit, vmax=local_limit)
    draw_field(axes[row, 2], u[sample_id, :, :, 0], f"first frame t={time_grid[0]:.2f}", theta[sample_id], vmin=-local_limit, vmax=local_limit)
    draw_field(axes[row, 3], u[sample_id, :, :, -1], f"last frame t={time_grid[-1]:.2f}", theta[sample_id], vmin=-local_limit, vmax=local_limit)

save_path = FIG_DIR / "train_gallery.png"
fig.savefig(save_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved figure -> {save_path}")


## Geometry And Dynamics Statistics

这里汇总每个样本的障碍物面积占比、孔径分布、孔心位置，以及轨迹的平均绝对涡量随时间变化。


In [ ]:
payload = datasets["train"]
u = np.asarray(payload["u"])
mask = np.asarray(payload["mask"]).astype(bool)
theta = np.asarray(payload["theta"])
time_grid = np.asarray(payload["t"]).reshape(-1)

mask_area_ratio = mask.mean(axis=(1, 2))
mean_abs_vorticity = np.abs(u).mean(axis=(1, 2))
radii = theta[:, :, 2].reshape(-1)
centers = theta[:, :, :2].reshape(-1, 2)

fig, axes = plt.subplots(1, 4, figsize=(20, 4.5), constrained_layout=True)

axes[0].hist(mask_area_ratio, bins=min(12, len(mask_area_ratio)), color="tab:blue", alpha=0.85)
axes[0].set_title("Obstacle Area Ratio")
axes[0].set_xlabel("mask mean")
axes[0].set_ylabel("count")

axes[1].hist(radii, bins=min(12, len(radii)), color="tab:orange", alpha=0.85)
axes[1].set_title("Hole Radius Distribution")
axes[1].set_xlabel("radius")
axes[1].set_ylabel("count")

for sample_curve in mean_abs_vorticity:
    axes[2].plot(time_grid, sample_curve, color="tab:blue", alpha=0.20)
axes[2].plot(time_grid, mean_abs_vorticity.mean(axis=0), color="black", lw=2.0, label="mean")
axes[2].set_title("Mean Absolute Vorticity")
axes[2].set_xlabel("time")
axes[2].set_ylabel("mean |w|")
axes[2].legend()

axes[3].scatter(centers[:, 0], centers[:, 1], s=1800.0 * radii**2, c=radii, cmap="viridis", alpha=0.7, edgecolors="black", linewidths=0.3)
axes[3].set_title("Hole Centers")
axes[3].set_xlabel("cx")
axes[3].set_ylabel("cy")
axes[3].set_xlim(0.0, 1.0)
axes[3].set_ylim(0.0, 1.0)
axes[3].set_aspect("equal")

save_path = FIG_DIR / "train_statistics.png"
fig.savefig(save_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved figure -> {save_path}")

stats = {
    "n_samples": int(mask.shape[0]),
    "resolution": int(mask.shape[1]),
    "record_steps": int(u.shape[-1]),
    "mask_area_ratio_mean": float(mask_area_ratio.mean()),
    "mask_area_ratio_min": float(mask_area_ratio.min()),
    "mask_area_ratio_max": float(mask_area_ratio.max()),
    "radius_mean": float(radii.mean()),
    "radius_min": float(radii.min()),
    "radius_max": float(radii.max()),
    "mean_abs_vorticity_final": float(mean_abs_vorticity[:, -1].mean()),
}
pprint(stats)


## Optional Auxiliary Fields

如果数据生成时启用了 `--save-aux-fields`，这里会额外画出速度和压力场；否则只给出提示。


In [ ]:
payload = datasets["train"]
required_keys = {"velocity_u", "velocity_v", "pressure"}

if not required_keys.issubset(payload):
    print("No auxiliary velocity/pressure fields found. Re-run the generator with --save-aux-fields to enable this section.")
else:
    sample_idx = 0
    frame_idx = -1
    mask = np.asarray(payload["mask"]).astype(bool)
    theta = np.asarray(payload["theta"])
    velocity_u = np.asarray(payload["velocity_u"])
    velocity_v = np.asarray(payload["velocity_v"])
    pressure = np.asarray(payload["pressure"])
    current_time = float(np.asarray(payload["t"]).reshape(-1)[frame_idx])

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5), constrained_layout=True)
    draw_field(axes[0], mask[sample_idx].astype(float), "mask", theta[sample_idx], cmap="gray_r", vmin=0.0, vmax=1.0)

    pressure_limit = max(float(np.max(np.abs(pressure[sample_idx, :, :, frame_idx]))), 1e-6)
    draw_field(axes[1], velocity_u[sample_idx, :, :, frame_idx], f"u_x at t={current_time:.2f}", theta[sample_idx])
    draw_field(axes[2], velocity_v[sample_idx, :, :, frame_idx], f"u_y at t={current_time:.2f}", theta[sample_idx])
    draw_field(axes[3], pressure[sample_idx, :, :, frame_idx], f"pressure at t={current_time:.2f}", theta[sample_idx], vmin=-pressure_limit, vmax=pressure_limit)

    save_path = FIG_DIR / f"train_sample_{sample_idx}_aux_fields.png"
    fig.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved figure -> {save_path}")


## Next Steps

- 如果这里只看到 `a/u/mask/theta/t`，说明当前数据已满足训练需要。
- 如果还要检查速度或压力边界行为，请用 `--save-aux-fields` 重新生成一小份 smoke 数据。
- 训练前建议先确认 `mask area ratio`、`radius distribution` 和末态 `mean |w|` 没有异常尖峰。
